In [ ]:
import torch
import matplotlib.pyplot as plt
%matplotlib widget

In [ ]:
# get the data set of 32x32 RGB images from the CIFAR-10 dataset
from torchvision import datasets
from torchvision import transforms

# define where the data will be stored
data_path = '../../data/data-unversioned/p1ch7/'

# download the training data
cifar10 = datasets.CIFAR10(data_path, train = True, download = True)

# download the validation data
cifar10_val = datasets.CIFAR10(data_path, train = False, download = True)

# define the class names as a list (corresponds to class)
class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

In [ ]:
# we have taken the means and variances from a previous notebook
cifar10 = datasets.CIFAR10(
    data_path, train=True, download=False,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4915, 0.4823, 0.4468),
                             (0.2470, 0.2435, 0.2616))
    ]))
cifar10_val = datasets.CIFAR10(
    data_path, train=False, download=False,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4915, 0.4823, 0.4468),
                             (0.2470, 0.2435, 0.2616))
    ]))

In [ ]:
# now we are going to subset the data
# to just consider birds and planes
label_map = {0:0, 2:1}
class_names = ['airplane','bird']
cifar2 = [(img, label_map[label]) for img, label in cifar10 if label in [0,2]]
cifar2_val = [(img, label_map[label]) for img, label in cifar10_val if label in [0,2]]

In [ ]:
# running the model based on random initialization
img, _ = cifar2[0]

plt.imshow(img.permute(1,2,0)) # place channels at end, as expected by imshow
plt.show()

In [ ]:
# running the model based on random initialization
img, _ = cifar2[5]

plt.imshow(img.permute(1,2,0)) # place channels at end, as expected by imshow
plt.show()

Note that some images are clear to the human eye, even in these standardized forms. However, some are far less distinguishable in these new transformed variables.

In [ ]:
# collect all the data into dimensions Samples x Features
# where Features is a collection of the CxHxW structure of the 
# original data format.
img_batch = img.view(-1).unsqueeze(0)

In [ ]:
# this is a single image, so rows are a single sample
img_batch.shape

In [ ]:
# DataLoader class helps with constructing batches from the full Dataset class
# DataLoader class helps with constructing batches from the full Dataset class
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=500, shuffle = True)
val_loader = torch.utils.data.DataLoader(cifar2_val, batch_size=500, shuffle = False)

In [ ]:
import torch.nn as nn
import torch.optim as optim

### Model Object ###

# define the model - here we try a linear model
model = nn.Sequential(
    nn.Linear(3072,2),
    nn.Tanh(),
    nn.LogSoftmax(dim=1)
)

### Optimization Objects ###
learning_rate = 1e-3

optimizer = optim.SGD(
    model.parameters(),
    lr = learning_rate
)

loss_fn = nn.NLLLoss()

### Optimization Loop ####
n_epochs = 1000

train_loss_data = []
val_loss_data = []

for epoch in range(1,n_epochs+1):
    accum_train_loss = 0
    for imgs, labels in train_loader:
        # calculate the train loss
        batch_size = imgs.shape[0]
        outputs = model(imgs.view(batch_size, -1))
        loss = loss_fn(outputs, labels)
        accum_train_loss += loss

        # perform optimization step on batch
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # store optimization information
    train_loss_data.append(accum_train_loss.item())

    # calculate validation loss for epoch
    with torch.no_grad():
        accum_val_loss = 0
        for imgs, labels in val_loader:
            # calculate the train loss
            batch_size = imgs.shape[0]
            outputs = model(imgs.view(batch_size, -1))
            loss = loss_fn(outputs, labels)
            accum_val_loss += loss
        
        # store optimization data
        val_loss_data.append(accum_val_loss.item())
    
    # update user as to progress during training 
    if epoch == 1 or epoch % 100 == 0:
        print(f"Epoch: {epoch}, Train Loss: {float(train_loss_data[-1])}, , Val Loss: {float(val_loss_data[-1])}")


In [ ]:
# calculate validation prediction errors for epoch
with torch.no_grad():
    num_correct = 0 
    num_false = 0
    num_total = 0
    for imgs, labels in val_loader:
        # calculate the train loss
        batch_size = imgs.shape[0]
        outputs = model(imgs.view(batch_size, -1))
        _, predicted_labels = torch.max(outputs, dim=1)
        # labels are a single row of Batch Size with some number
        # outputs are a single row of 2D. So, retrieve the max
        # value and output into a single

        num_correct += sum(predicted_labels == labels)
        num_false += sum(predicted_labels != labels)
        num_total += batch_size
    
print(f"Precision: {num_correct/num_total}")
print(f"Number Total: {num_total}")

In [ ]:
import numpy as np

plt.figure()
plt.plot(train_loss_data,label = "Train")
plt.plot(val_loss_data, label = "Validation")
plt.xlabel("Epochs")
plt.ylabel('Loss)')
plt.legend()
plt.show()

In [ ]:
import numpy as np
tld = np.array(train_loss_data)
vld = np.array(val_loss_data)
plt.figure()
plt.plot((tld - tld.min())/(tld.max()-tld.min()),label = "Train")
plt.plot((vld - vld.min())/(vld.max()-vld.min()), label = "Validation")
plt.xlabel("Epochs")
plt.ylabel('Normalized Loss')
plt.legend()
plt.show()

In [ ]:
# check the number of parameters
numel_list = [p.numel() for p in model.parameters() if p.requires_grad == True]

sum(numel_list), numel_list

In [ ]:
model

In [ ]:
plt.figure()
plt.hist(model[0].weight.tolist(),bins = 20)
plt.legend(['Node 1', 'Node 2'])
plt.show()

Most are linear and so we can probably remove a lot of the parameters. That being said, most parameters are small anyways, so this implies that "nearly" zero is only relative here. 